<a href="https://colab.research.google.com/github/kellieoquinn-idir/TLab-Phase-2---Building-a-Customer-Chat-Bot/blob/main/customer_service_rag_assignment%5B1%5D.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Build a Customer Service RAG Chatbot

**Goal:** Build a working customer service chatbot that uses Retrieval-Augmented
Generation (RAG) to answer questions grounded in a knowledge base of 10
documents, powered by the DeepSeek LLM and ChromaDB. The chatbot should also
remember earlier turns in the conversation, so follow-up questions work
naturally. Then, wrap it in a simple Streamlit web UI.

This notebook is your guide. It will not give you finished code for every
step — you'll fill in the pieces marked `# TODO`, using what you learned in
class as a reference. Each section ends with a **✅ Checkpoint** telling you
what to check before moving on.

## What you'll build
1. A knowledge base of 10 customer service documents (for a company you choose)
2. A ChromaDB vector store to search that knowledge base
3. A retrieval function
4. A connection to the DeepSeek LLM
5. A `chat()` function that ties retrieval + generation together **and**
   carries conversation history forward, so the bot stays aware of what was
   already discussed
6. A Streamlit UI so people can actually talk to your chatbot in a browser,
   with real multi-turn memory

## Setup
Install the packages you'll need (DON'T FORGET TO SETUP A VIRTUAL ENVIRONMENT):
```
pip install chromadb scikit-learn openai python-dotenv streamlit
```

Create a `.env` file in your project folder with the following keys:
```
DEEPSEEK_API_KEY=your-key-here
CHROMADB_API_KEY=your-key-here
CHROMADB_TENANT=your-tenant-id-here
CHROMADB_DB=your-chroma-db-name-here
```
Replace the values with your actual keys and other information

**This assignment requires internet access**

In [ ]:
import os
import chromadb
from sklearn.feature_extraction.text import TfidfVectorizer
from dotenv import load_dotenv
from openai import OpenAI

load_dotenv()

print("Imports OK" if os.environ.get("DEEPSEEK_API_KEY") else
      "Imports OK, but no DEEPSEEK_API_KEY found -- check your .env file")

ModuleNotFoundError: No module named 'chromadb'

**✅ Checkpoint 1:** Running the cell above should print `Imports OK` and
confirm your API key was found. If you see an import error, go back and
`pip install` the missing package.

## Part 1: Build your knowledge base

Pick a fictional company and write **10 short customer-service documents**
for it -- things like return policy, shipping times, warranty, payment
methods, how to contact support, etc. (The example below is for a fictional
electronics company, "BrightPath Electronics" -- feel free to use it as a
starting point, but you should customize it or write your own from scratch.)

**Requirements:**
- Exactly 10 documents
- Each one should be a self-contained fact or policy (1-3 sentences)
- Cover a *range* of topics (returns, shipping, payments, support hours, etc.) so retrieval has something meaningful to differentiate between

In [ ]:
documents = [
    # TODO: Replace these with your own 10 customer-service documents,
]

assert len(documents) == 10, f"You need exactly 10 documents, you have {len(documents)}"
doc_ids = [f"doc_{i}" for i in range(len(documents))]

for i, doc in enumerate(documents):
    print(f"[{doc_ids[i]}] {doc}")

**✅ Checkpoint 2:** The assertion should pass silently (no error), and you
should see all 10 of your documents printed above.

## Part 2: Build the ChromaDB vector store

This part is the same pattern from class: fit a TF-IDF vectorizer on your
documents, then store the documents + their embeddings in a ChromaDB
collection. Fill in the TODOs below using what you built in the in-class demo.

In [ ]:
def build_vector_store(documents, doc_ids):
    """Create a ChromaDB collection and populate it with the given documents.

    Returns: (collection, vectorizer)
    """
    client = chromadb.CloudClient(
        api_key="REPLACE WITH YOUR CHROMA DB API KEY USING THE 'os' MODULE",
        tenant="REPLACE WITH YOUR CHROMA DB TENANT ID USING THE 'os' MODULE",
        database="REPLACE WITH YOUR CHROMA DB DATABASE NAME USING THE 'os' MODULE"
    )

    # TODO: create a collection. Remember to pass embedding_function=None
    # since we're supplying our own TF-IDF embeddings.
    collection = None  # <-- replace this

    # TODO: create a TfidfVectorizer, fit it on `documents`, and convert
    # the result to a list of lists (.toarray().tolist())
    vectorizer = None       # <-- replace this
    doc_embeddings = None   # <-- replace this

    # TODO: add the documents, embeddings, and ids to the collection

    return collection, vectorizer


collection, vectorizer = build_vector_store(documents, doc_ids)
print(f"Stored {collection.count()} documents in collection '{collection.name}'")

**✅ Checkpoint 3:** This should print `Stored 10 documents in collection
'...'`. If you get an `AttributeError: 'NoneType' object has no attribute`,
you still have a TODO left unfilled above.

## Part 3: Write a retrieval function

Write a function that takes a question, embeds it the same way you embedded
your documents, and returns the top matching documents from your collection.

In [ ]:
def retrieve(collection, vectorizer, question, n_results=3):
    """Return the n_results documents most relevant to `question`."""
    # TODO:
    #   1. Embed the question using vectorizer.transform([question])
    #   2. Query the collection with that embedding
    #   3. Return results["documents"][0]
    pass  # <-- replace this


# Try it out
test_question = "How long do I have to return something?"
retrieved = retrieve(collection, vectorizer, test_question)
print(f"Question: {test_question}")
for doc in retrieved:
    print(" -", doc)

**✅ Checkpoint 4:** For the sample question above, your top result should
be the return-policy document (or whichever of your own documents covers
returns). If `retrieved` is `None` or empty, check your `retrieve()` function.

## Part 4: Connect to DeepSeek and generate an answer

Now write the pieces that turn retrieved documents into a real,
LLM-generated answer. We're splitting this into three pieces instead of two,
specifically so the model can be given conversation history later in Part 5:

1. `SYSTEM_PROMPT` -- a constant string describing the assistant's role and
   rules (customer service assistant for YOUR company, answer ONLY from
   context, say "I don't know" otherwise). This doesn't change turn to turn,
   so it's defined once instead of being rebuilt inside a function.
2. `build_prompt(question, retrieved_docs)` -- formats the context + question
   for **just the current turn** into a single string. It no longer returns
   the system message -- that's handled separately now, since the system
   message should appear once per conversation, not once per turn.
3. `generate_answer(messages)` -- sends a full list of `{"role": ..., "content": ...}`
   messages to DeepSeek and returns the response text. Taking a full message
   list (instead of a single prompt string) is what will let us hand the
   model prior conversation turns in Part 5.

In [ ]:
# TODO: write a SYSTEM_PROMPT constant that:
#   - tells the model it's a customer service assistant for YOUR company
#   - instructs it to answer using ONLY the context provided
#   - instructs it to say it doesn't know (and suggest contacting support)
#     if the answer isn't in the context
# This is defined once, since it doesn't change turn to turn.
SYSTEM_PROMPT = None  # <-- replace this


def build_prompt(question, retrieved_docs):
    """Build the CURRENT TURN's user-message content (context + question).

    This only needs to cover what's relevant to THIS question -- the system
    message above is handled separately in chat() (Part 5), and conversation
    history will get threaded in there too.
    """
    context = "\n".join(f"- {doc}" for doc in retrieved_docs)

    # TODO: build a single string that includes both `context` and `question`
    prompt = None  # <-- replace this

    return prompt


def generate_answer(messages):
    """Send a full list of conversation messages to DeepSeek and return the
    response text.

    `messages` is a list of {"role": ..., "content": ...} dicts, e.g.:
    [{"role": "system", "content": ...}, {"role": "user", "content": ...}]
    Taking the FULL list (instead of a single prompt string) is what lets us
    hand the model prior conversation turns in Part 5.
    """
    api_key = "REPLACE WITH YOUR DEEPSEEK API KEY USING THE 'os' MODULE"
    if not api_key:
        return "(No DEEPSEEK_API_KEY found -- check your .env file)"

    # TODO:
    #   1. Create an OpenAI client with base_url="https://api.deepseek.com"
    #   2. Call client.chat.completions.create(...) with model="deepseek-v4-flash"
    #      and `messages` as the messages argument
    #   3. Return response.choices[0].message.content
    pass  # <-- replace this


# Try it out (single turn, no history yet)
prompt = build_prompt(test_question, retrieved)
messages = [
    {"role": "system", "content": SYSTEM_PROMPT},
    {"role": "user", "content": prompt},
]
answer = generate_answer(messages)
print(answer)

**✅ Checkpoint 5:** You should see a real, generated answer about your
return policy, in the voice of a customer service assistant. If you see the
"(No DEEPSEEK_API_KEY found...)" message, check your `.env` file.

## Part 5: Put it all together into one `chat()` function, with memory

Combine everything above into a single function you'll reuse in your
Streamlit app. This time, `chat()` also needs an optional `history` argument:
a list of prior `{"role": ..., "content": ...}` messages from earlier in the
conversation (not including the current question).

Given a question (and optionally some history), `chat()` should:
1. Retrieve context for the current question (retrieval itself still only
   looks at the current question, not the full history)
2. Build the current turn's message content with `build_prompt()`
3. Assemble the full message list: `[system message] + history + [current turn]`
4. Call `generate_answer()` with that full list and return the answer

Passing `history` in is what makes follow-up questions like "how long does
that take?" work -- without it, the model only ever sees one question in
isolation and has no way to know what "that" refers to.

In [ ]:
def chat(collection, vectorizer, question, history=None, n_results=3):
    """End-to-end RAG with conversation memory.

    `history` is an optional list of prior {"role": ..., "content": ...}
    messages (e.g. from st.session_state.messages), NOT including the
    current `question`. Passing `history` in is what makes follow-up
    questions like "how long does that take?" work.
    """
    # TODO:
    #   1. Call retrieve() to get relevant docs for `question`
    #   2. Call build_prompt() to build the CURRENT TURN's content
    #   3. Assemble messages = [system message] + (history or []) + [current turn]
    #   4. Call generate_answer() with that full messages list
    #   5. Return the answer
    pass  # <-- replace this




# Test: a follow-up question that ONLY makes sense with conversation history.
# TODO: replace these with a real pair from YOUR knowledge base -- a first
# question, then a follow-up that only makes sense if the bot remembers the
# first (e.g. "Do you do custom builds?" then "How long does that take?")
print("\n" + "="*70)
print("MULTI-TURN TEST")

turn_1_question = "PROVIDE YOUR FIRST QUESTION HERE"
turn_1_answer = chat(collection, vectorizer, turn_1_question)
print("Q1:", turn_1_question)
print("A1:", turn_1_answer)

history = [
    {"role": "user", "content": turn_1_question},
    {"role": "assistant", "content": turn_1_answer},
]

turn_2_question = "PROVIDE A FOLLOW-UP QUESTION HERE"
turn_2_answer = chat(collection, vectorizer, turn_2_question, history=history)
print("\nQ2:", turn_2_question)
print("A2:", turn_2_answer)

**✅ Checkpoint 6:** The first three questions should behave as expected (two
grounded answers, one honest "I don't know"). For the **multi-turn test**:
your second question should get correctly answered using information from
the first turn, even if the second question alone doesn't repeat the key
topic word. If Q2's answer seems confused or off-topic, double check that
`chat()` is passing `history` into the `messages` list.

## Part 6: Move your working code into `llm_with_rag.py`

Streamlit apps run as separate standalone scripts -- they don't share this
notebook's kernel. So before building the UI, copy your **finished, tested**
code into a new file called `llm_with_rag.py`. There is already a template file there named "llm_with_rag_template.py". Just copy and paste that file in the same directory and rename it to: `llm_with_rag.py` in your project folder. There is already some starting functions in there.

Make sure `llm_with_rag.py` includes the `SYSTEM_PROMPT` constant and the
updated `chat(collection, vectorizer, question, history=None, n_results=3)`
signature -- your Streamlit app will rely on the `history` parameter to give
the bot conversation memory.

**✅ Checkpoint 7:** In a terminal, run:
```
python -c "from llm_with_rag import chat, build_vector_store, documents, doc_ids; c, v = build_vector_store(documents, doc_ids); print(chat(c, v, 'What is your return policy?'))"
```
You should see a real generated answer print out. If you get an
`ImportError`, double check your `llm_with_rag.py` is saved in the same folder you're
running the command from.

## Part 7: Build the Streamlit UI

Now you'll create and build an `app.py` in stages. Copy each snippet below into your `app.py`,
save, and run:
```
streamlit run app.py
```
Make sure you are running and testing the code **BEFORE** moving onto the next stage.

Note: conversation memory doesn't show up until **Stage 3**. Stages 1-2 are
single-turn (each question is independent) so you can confirm the basics
work first before adding history on top.

### Stage 1 -- basic skeleton

Create `app.py` with just this to start:

In [ ]:
import streamlit as st

st.title("💬 Customer Support Chat")
st.write("<WRITE YOUR OWN DESCRIPTION FOR THE BOT YOURSELF>")

question = st.text_input("Your question:")

if question:
    st.write(f"You asked: {question}")

**✅ Checkpoint 8a:** Run `streamlit run app.py`. A browser tab should open
showing your title. Type something in the box and confirm it echoes back
"You asked: ...". This just confirms Streamlit itself is working -- no RAG yet.

### Stage 2 -- wire in your RAG backend

Replace the contents of `app.py` with this:

In [ ]:
import streamlit as st
from llm_with_rag import build_vector_store, chat, documents, doc_ids

st.title("💬 Customer Support Chat")
st.write("<WRITE YOUR OWN DESCRIPTION FOR THE BOT YOURSELF>")

# Build the vector store once and cache it, instead of rebuilding it on
# every single interaction (Streamlit reruns your whole script on each action!)
@st.cache_resource
def get_collection():
    return build_vector_store(documents, doc_ids)

collection, vectorizer = get_collection()

question = st.text_input("Your question:")

if question:
    with st.spinner("Thinking..."):
        answer = chat(collection, vectorizer, question)
    st.write(answer)

Note: this stage doesn't pass any `history` to `chat()` yet, so each question
is still answered in isolation -- that's expected here, memory comes in
Stage 3.

**✅ Checkpoint 8b:** Ask a real question (e.g. "What's your return policy?").
You should get a grounded answer generated by DeepSeek. Try asking something
unrelated too, and confirm you get an honest "I don't know" instead of a
made-up answer.

### Stage 3 -- add conversation history (chat bubbles)

Replace the contents of `app.py` with this:

In [ ]:
import streamlit as st
from llm_with_rag import build_vector_store, chat, documents, doc_ids

st.title("💬 Customer Support Chat")
st.write("<WRITE YOUR OWN DESCRIPTION FOR THE BOT YOURSELF>")

@st.cache_resource
def get_collection():
    return build_vector_store(documents, doc_ids)

collection, vectorizer = get_collection()

# Keep track of the conversation across reruns
if "messages" not in st.session_state:
    st.session_state.messages = []

# Redraw the whole conversation so far
for msg in st.session_state.messages:
    with st.chat_message(msg["role"]):
        st.write(msg["content"])

# st.chat_input pins a text box to the bottom of the page, like a real chat app
if question := st.chat_input("Ask a question..."):
    st.session_state.messages.append({"role": "user", "content": question})
    with st.chat_message("user"):
        st.write(question)

    with st.chat_message("assistant"):
        with st.spinner("Thinking..."):
            # st.session_state.messages[:-1] is everything BEFORE this new
            # question -- passing it as `history` is what gives the bot
            # memory of the conversation so far.
            answer = chat(collection, vectorizer, question, history=st.session_state.messages[:-1])
        st.write(answer)
    st.session_state.messages.append({"role": "assistant", "content": answer})


**✅ Checkpoint 8c:** Ask two or three questions in a row. Confirm all of
them stay visible on screen as a scrolling conversation, not just the latest
one. Then test the memory specifically: ask a question, then a follow-up
that only makes sense with the first question in mind (e.g. "Do you do
custom builds?" followed by "How long does that take?") -- the second answer
should correctly address it without you having to repeat the topic yourself.

### Stage 4 -- show what was retrieved

It's often useful (and reassuring to users) to show *which* source
documents an answer was based on. Add this inside the `with
st.chat_message("assistant"):` block, right after computing `answer`:

In [ ]:
retrieved_docs = retrieve(collection, vectorizer, question)
with st.expander("Sources used"):
    for doc in retrieved_docs:
        st.write(f"- {doc}")

Don't forget to add `retrieve` to your import line at the top:

In [ ]:

from llm_with_rag import build_vector_store, chat, retrieve, documents, doc_ids


**✅ Checkpoint 8d:** Ask a question and confirm an expandable
"Sources used" section appears showing the actual documents that were
retrieved for that answer.

At the end, your customer service bot should look somethign close to this (everyone's bot might vary based on the fake business you created):

![Screenshot](Screenshot_2026-07-28_15-28-52.png)